## Introduction

Ce Notebook presente le code pour récuperer les données nécessaires pour analyser les emissions dans la région l'île de France. Pour étudier notre problématique rigoureusement, on doit utiliser des données composant:

1) On a besoin des concentrations de particules qui puissent nuire à la population. On utilise le NO2, PM2,5 et le O3. On note que la récupération de ses données n'est pas si directe comme on pourrait l'imaginer. En effet, le site the Airparif (qui est le site donnant les données sur la qualité de l'air de toute la région de l'île de France) ne donne que les valeurs des stations qui mesurent la qualité de l'air. Or, on a besoin de pouvoir une vision spatiale de toutes les communes et pas que de certaines stations. Alors, la récupération de ces données devient complexe. Néanmoins, Airparifs offre aussi des cartes intéractives qui donnent les niveaux de concentrations des particules. Donc, en utilisant ces cartes avec les données déjà modélisées sur toute l'Île de France, on pourrait réaliser notre étude. J'ai inspectionné le Slider donnant les cartes et j'ai trouvé d'oú je pouvais extraire les données. 

2) Pour pouvoir utiliser les données on devra regrouper les données par communes de l'INSEE, donc on a besoin des coordonnées de centroïde de chaque commune. Pour cela on utilise la geo API de datagouv.fr, qui donne des coordonnées des communes.

L'idée est d'utiliser les coordonées des centroïdes des communes et de "scraper" la concentration des particules nuisant la qualité de l'air directement des données qui sont presentées sur la carte (qui modélisent directement la concentration des particules aux alentours des stations de mesure)


In [1]:
import numpy as np
import pandas as pd
import requests
from functions_emissions import extract_values 
import s3fs

### Scraper les données des cartes de Airparif

In [2]:
# Données du Geoserver de Airparif 

url_base = "https://namek.airparif.fr/geoserver/Moyenne_annuelle/ows?service=WMS&version=1.3.0&request=GetFeatureInfo"
years = range(2007,2025)
particles = {"no2","o3","pm25"}


In [3]:
# Données des centroïdes des communes de l'île de france

url_communes = "https://geo.api.gouv.fr/communes"
departement_idf = ["75","77","78","91","92","93","94","95"]

data_centroides = []

for dep in departement_idf:
    parametres = {"codeDepartement": dep, "type": "commune-actuelle,arrondissement-municipal","fields": "nom,code,codeDepartement,centre", "format": "json"}
    r = requests.get(url_communes, params = parametres).json()
    data_centroides += r
    
ile_de_fr_commune = pd.DataFrame({
    "INSEE_Commune" : [com["code"] for com in data_centroides],
    "INSEE_Departement" : [com["codeDepartement"] for com in data_centroides],
    "Nom_Commune" : [com["nom"] for com in data_centroides],
    "Latitude" : [com["centre"]["coordinates"][1] for com in data_centroides],
    "Longitude" : [com["centre"]["coordinates"][0] for com in data_centroides]})

ile_de_fr_commune = ile_de_fr_commune.drop(index=0).reset_index(drop=True) # la premiere ligne est Paris en general, donc on l'enleve

In [ ]:
# Scraper les données de concentration pour chaque commune dans l'île-de-France (Cela a pris approx 100 min)
data_concentration = []

for communes in range(ile_de_fr_commune.shape[0]):
    for particule in particles:
        for year in years:
            concentration = extract_values(url_base, ile_de_fr_commune["Longitude"].loc[communes], ile_de_fr_commune["Latitude"].loc[communes], particule, year)
            dict_year_particle = {"Year": year, "Particle": particule, "Commune": ile_de_fr_commune["Nom_Commune"].loc[communes], "Code_INSEE": ile_de_fr_commune["INSEE_Commune"].loc[communes], "Departement": ile_de_fr_commune["INSEE_Departement"].loc[communes],"Concentration": concentration}
            data_concentration.append(dict_year_particle)


In [5]:
# Construction du Panel (Commune, année, Partibule_polluante) pour l'analyse
Data_Emissions = pd.DataFrame(data_concentration).pivot_table(index=["Commune", "Code_INSEE", "Departement", "Year"], columns= "Particle", values= "Concentration")

In [ ]:
# Stocker les données sur SSP Cloud

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/concentrations_idf.parquet", "wb") as f:
    Data_Emissions.to_parquet(f)


## Étude et Description des Données

Maintenant que on a le panel de données avec toutes les particules par commune par année, on peut commencer à faire l'analyse des statistiques des données. On structure cette section de la façon suivante:

1) Tout d'abord, on regarde de manière générale la distribution des valeurs 
2) Analyse geographique: Où trouvons-nous les zones les plus polluées?
3) Analyse Temporelle: Voit-on des dynamiques? Peut-on observer des shocks importants?

In [2]:
# Ouvrir le parquet directement de SSpCloud

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/concentrations_idf.parquet", "rb") as f:
    Data_Emissions = pd.read_parquet(f)

### Description des données



#### Problèmes avec les données

Tout d'abord, on s'interesse à la structure des données poour voir si il y a eu des problèmes dans l'extraction des données. Ici, on peut voir qu'il y a un problème avec les données de certaines communes. Donc, on exlore quelle est la source de ce problème en regardant quelles sont les communes donnant ces valeures abérrantes. On peut voir que le problème vient que de la commune de Saint-Pathus, et que en l'enlevant de notre échantillon, on obtient des valeurs beaucoup plus raisonables. 

Les données de cette communes sont l'équivalent à des données NA puisque elles n'ont aucune validité. Pour ne pas corrompre notre analyse, on excluira cette commune. Cela est raisonable puisque c'est une petitte commune frontalière de l'île de france, et donc, l'exclusion n'aura pas de grandes conséquences sur nos résultats.

In [3]:
Data_Emissions.describe()

/opt/python/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/opt/python/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


Particle,no2,o3,pm25
count,23129.000000,2.312900e+04,2.312800e+04
mean,9.059044,inf,inf
std,263.460748,inf,inf
min,-9999.000000,-9.999000e+03,-9.999000e+03
25%,9.666734,7.769344e+00,8.835938e+00
50%,14.645324,1.168689e+01,1.130522e+01
75%,20.705441,1.724444e+01,1.402599e+01
max,69.558527,1.234500e+308,1.234500e+308


In [4]:
Data_Emissions[Data_Emissions["o3"] >= 100]

Particle                                     no2             o3           pm25
Commune      Code_INSEE Departement Year                                      
Saint-Pathus 77430      77          2007 -9999.0  1.234500e+308  -9.999000e+03
                                    2008 -9999.0  1.234500e+308  -9.999000e+03
                                    2009 -9999.0  1.234500e+308  -9.999000e+03
                                    2010 -9999.0  1.234500e+308  -9.999000e+03
                                    2011 -9999.0  1.234500e+308  -9.999000e+03
                                    2012 -9999.0  1.234500e+308  -9.999000e+03
                                    2013 -9999.0  1.234500e+308  -9.999000e+03
                                    2014 -9999.0  1.234500e+308  -9.999000e+03
                                    2018 -9999.0  1.234500e+308  1.234500e+308
                                    2019 -9999.0   1.234500e+38   1.234500e+38
                                    2020 -9999.0  1.234500e+308   1.234500e+38
                                    2021 -9999.0  1.234500e+308   1.234500e+38

In [7]:
Data_Emissions = Data_Emissions.drop("Saint-Pathus", level="Commune")
Data_Emissions.describe()

Particle,no2,o3,pm25
count,23112.000000,23112.000000,23112.000000
mean,15.987309,12.591292,11.770842
std,8.478551,6.656286,3.476377
min,2.177196,0.000000,6.062500
25%,9.673809,7.769270,8.835938
50%,14.652687,11.684671,11.306413
75%,20.713225,17.238471,14.021851
max,69.558527,39.922986,31.783500


#### Description géographique des données

Maintenant qu'on a une meilleur idée de la forme de nos données, on peut aller plus loin sur l'analyse.
On va commencer par une étude departementale pour voir si on peut déjà expliquer les differences en concentrations.

In [10]:
Data_Emissions_Dep = Data_Emissions.groupby(level="Departement").mean()
Data_Emissions_Dep

Particle,no2,o3,pm25
Departement,,,
75,37.460871,9.291148,15.364768
77,12.016128,11.467751,11.342181
78,14.773647,12.931300,11.256636
91,17.241009,14.291408,11.598466
92,28.520091,11.116324,13.808036
93,30.219176,12.091371,14.084041
94,27.778445,12.022074,13.253106
95,16.482724,14.317954,12.199385
